# Notebook 05 — Testing Workflows

This notebook covers testing Temporal Workflows and Activities using `ActivityEnvironment` and `WorkflowEnvironment`.

## What You Will Learn
- How to test Activities in isolation
- How to test Workflows with a test server
- Time skipping for fast tests
- Mocking Activities for unit tests

## Testing Activities with ActivityEnvironment

`ActivityEnvironment` runs Activities outside of Temporal, providing the Activity context.

```python
from temporalio.testing import ActivityEnvironment

activity_env = ActivityEnvironment()
result = await activity_env.run(my_activity, input_arg)
assert result == expected
```

In [ ]:
from dataclasses import dataclass
from temporalio import activity
from temporalio.testing import ActivityEnvironment


@dataclass
class Address:
    line1: str
    line2: str
    city: str
    state: str
    postal_code: str


@dataclass
class Distance:
    kilometers: int


class PizzaActivities:
    @activity.defn
    async def get_distance(self, address: Address) -> Distance:
        kilometers = len(address.line1) + len(address.line2) - 10
        if kilometers < 1:
            kilometers = 5
        return Distance(kilometers=kilometers)


# Test the activity
async def test_get_distance():
    env = ActivityEnvironment()
    activities = PizzaActivities()
    
    address = Address(
        line1="701 Mission Street",
        line2="Apartment 9C",
        city="San Francisco",
        state="CA",
        postal_code="94104",
    )
    
    result = await env.run(activities.get_distance, address)
    print(f"Distance: {result.kilometers} km")
    assert result == Distance(20)
    print("✓ Test passed!")


await test_get_distance()

## Parametrized Tests

Use `pytest.mark.parametrize` to test multiple cases:

```python
@pytest.mark.asyncio
@pytest.mark.parametrize("input, output", [
    (Address(line1="701 Mission", line2="Apt 9C", ...), Distance(20)),
    (Address(line1="917 Delores", line2="", ...), Distance(8)),
])
async def test_get_distance(input, output):
    env = ActivityEnvironment()
    activities = PizzaActivities()
    assert output == await env.run(activities.get_distance, input)
```

## Testing Workflows with WorkflowEnvironment

`WorkflowEnvironment` starts a lightweight Temporal server for testing:

```python
from temporalio.testing import WorkflowEnvironment
from temporalio.worker import Worker

async with await WorkflowEnvironment.start_time_skipping() as env:
    async with Worker(
        env.client,
        task_queue="test-queue",
        workflows=[MyWorkflow],
        activities=[my_activity],
    ):
        result = await env.client.execute_workflow(
            MyWorkflow.run, input,
            id="test-id", task_queue="test-queue",
        )
        assert result == expected
```

## Time Skipping

`start_time_skipping()` automatically advances time past `asyncio.sleep()` calls.

A Workflow with `await asyncio.sleep(86400)` (24 hours) completes **instantly** in tests.

## Mocking Activities

Replace real Activities with mocks for unit testing:

```python
@activity.defn(name="translate_term")
async def translate_term_mocked(input):
    if input.term == "hello":
        return TranslationActivityOutput("Bonjour")
    return TranslationActivityOutput("Au revoir")


async with Worker(
    env.client,
    task_queue="test-queue",
    workflows=[TranslationWorkflow],
    activities=[translate_term_mocked],  # Mock instead of real
):
    ...
```

The `name=` parameter must match the original Activity's registered name.

## Testing Error Cases

```python
from temporalio.client import WorkflowFailureError
from temporalio.exceptions import ApplicationError

with pytest.raises(WorkflowFailureError) as e:
    await env.client.execute_workflow(...)

assert isinstance(e.value.cause, ApplicationError)
assert "error message" == str(e.value.cause)
```

In [ ]:
# Test error handling
from temporalio.exceptions import ApplicationError


@dataclass
class Bill:
    customer_id: int
    order_number: str
    description: str
    amount: int


class BillingActivities:
    @activity.defn
    async def send_bill(self, bill: Bill) -> str:
        if bill.amount < 0:
            raise ApplicationError(f"invalid charge amount: {bill.amount}")
        return "SUCCESS"


async def test_send_bill_negative_amount():
    env = ActivityEnvironment()
    activities = BillingActivities()
    bill = Bill(customer_id=1, order_number="X1", description="test", amount=-500)
    
    try:
        await env.run(activities.send_bill, bill)
        print("✗ Should have raised an error")
    except ApplicationError as e:
        print(f"✓ Caught expected error: {e}")


await test_send_bill_negative_amount()

## Running Tests with pytest

```bash
# Run all tests
pytest -v

# Run specific test file
pytest tests/test_activities.py -v

# Run specific test
pytest tests/test_activities.py::test_get_distance -v
```

## Summary

| Tool | Use Case |
|------|----------|
| `ActivityEnvironment` | Test Activities in isolation |
| `WorkflowEnvironment.start_time_skipping()` | Test Workflows with fast timers |
| `@activity.defn(name=...)` | Mock Activities for unit tests |
| `pytest.raises(WorkflowFailureError)` | Test error cases |

**Next:** [06_debugging_with_event_history.ipynb](06_debugging_with_event_history.ipynb)